# 01. Limpieza y Preparación de Datos (ETL) — Healthcare Operations
Este notebook documenta la auditoría de calidad de datos, tratamiento de valores nulos y duplicados, estandarización de fechas e ingeniería de variables financieras (`tipo_atencion_estancia` y `financial_risk_impact`) para generar el dataset analítico listo para SQL y Business Intelligence.

In [ ]:
import pandas as pd
import numpy as np

# Carga del dataset original
df = pd.read_csv('../data/Original_healthcare.csv', encoding='latin-1')
print(f'Dimensiones iniciales: {df.shape}')

In [ ]:
# Revisión general de los datos
display(df.head(5))
df.info()
display(df.describe())

In [ ]:
# Auditoría de valores nulos y duplicados
print('Valores nulos por columna:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'Registros duplicados: {df.duplicated().sum()}')

In [ ]:
# Conversión de fechas y estandarización de cadenas de texto
df['visit_date'] = pd.to_datetime(df['visit_date'], errors='coerce')

text_cols = df.select_dtypes(include=['object', 'string']).columns
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())

In [ ]:
# Ingeniería de variables de negocio (Feature Engineering)
# 1. Tipo de atención por días de estancia (Ambulatorio si es 0, Hospitalizado en otro caso)
df['tipo_atencion_estancia'] = df['length_of_stay_days'].apply(
    lambda x: 'Ambulatorio / Alta el mismo día' if x == 0 else 'Hospitalizado'
)

# 2. Impacto de Riesgo Financiero (Costo de Tratamiento * Riesgo de Readmisión)
df['financial_risk_impact'] = df['treatment_cost'] * df['readmission_risk']

print('--- Resumen de Estancias ---')
print(df['tipo_atencion_estancia'].value_counts())
print(f'Dimensiones finales limpias: {df.shape}')

In [ ]:
# Exportación del dataset limpio listo para MySQL y Power BI / Excel
df.to_csv('../data/Healthcare_Cleaned_v2.csv', index=False, encoding='utf-8')
print('Archivo Healthcare_Cleaned_v2.csv exportado correctamente.')